# SOTA CPT Dataset Prep v2 (Notebook A_sota)

Builds Hugging Face datasets from the multi-source theology mix:

```bash
python continued_pretrain/scripts/07_build_theology_mix.py
python continued_pretrain/scripts/06_verify_tokens.py --mix
```

**Does not replace** `A_data_prep.ipynb` (Spurgeon-only).

### Guards (G2)
Refuses to build if the mix is Spurgeon-only (single domain bucket) unless
`ALLOW_SPURGEON_ONLY = True` (diagnostics only).

### Outputs
- `/kaggle/working/theology_dataset` (train/test)
- `/kaggle/working/theology_holdouts/{spurgeon,puritan,confession,general}`


## 1. Install

In [ ]:
!pip install datasets -q

## 2. Config

In [ ]:
import os
import json
from pathlib import Path
from datasets import Dataset, DatasetDict

CORPUS_ROOT = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-corpus"
TRAIN_TXT = os.path.join(CORPUS_ROOT, "theology_mix_train.txt")
HOLDOUT_DIR = os.path.join(CORPUS_ROOT, "holdouts")
MANIFEST_PATH = os.path.join(CORPUS_ROOT, "theology_mix_manifest.json")

OUT_TRAIN = "/kaggle/working/theology_dataset"
OUT_HOLDOUTS = "/kaggle/working/theology_holdouts"
DOC_SEP = "<|endoftext|>"
MIN_CHARS = 200
VAL_FRACTION = 0.01
SEED = 42
ALLOW_SPURGEON_ONLY = False  # G2: True only for diagnostics

print("Config OK")

## 3. G2 multi-bucket guard + parse training mix

In [ ]:
def parse_concat_txt(path, min_chars=MIN_CHARS):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    text = Path(path).read_text(encoding="utf-8")
    docs = [d.strip() for d in text.split(DOC_SEP) if len(d.strip()) > min_chars]
    print(f"{path}: {len(text):,} chars -> {len(docs)} docs")
    max_doc = max((len(d) for d in docs), default=0)
    over = sum(1 for d in docs if len(d) > 8000)
    print(f"  max_doc_chars={max_doc}  docs>8k={over}")
    return docs

# G2: refuse Spurgeon-only "SOTA" mixes
if os.path.exists(MANIFEST_PATH):
    manifest = json.loads(Path(MANIFEST_PATH).read_text(encoding="utf-8"))
    buckets = manifest.get("buckets") or {}
    domain = [b for b in buckets if b in ("spurgeon", "puritan", "confession", "bible")]
    non_empty = [b for b in domain if (buckets.get(b) or {}).get("chars", 0) > 0]
    print("Manifest domain buckets with chars:", non_empty)
    print("Bucket shares:", {b: buckets[b].get("char_share") for b in buckets})
    if len(non_empty) < 2 and not ALLOW_SPURGEON_ONLY:
        raise RuntimeError(
            "G2: theology mix has <2 domain buckets "
            f"{non_empty}. Add Puritans/confessions/Bible and rebuild with "
            "07_build_theology_mix.py (omit --allow-spurgeon-only). "
            "Set ALLOW_SPURGEON_ONLY=True only for diagnostics."
        )
else:
    print("WARNING: no manifest found; cannot enforce multi-bucket guard.")

train_docs = parse_concat_txt(TRAIN_TXT)
train_ds = Dataset.from_dict({"text": train_docs})
split = train_ds.train_test_split(test_size=VAL_FRACTION, seed=SEED)
print(split)
print(f"train={len(split['train'])} val={len(split['test'])}")

## 4. Parse multi-holdouts

In [ ]:
holdout_names = ["spurgeon", "puritan", "confession", "general"]
holdouts = {}

for name in holdout_names:
    p = os.path.join(HOLDOUT_DIR, f"{name}_holdout.txt")
    if os.path.exists(p):
        docs = parse_concat_txt(p)
        if docs:
            holdouts[name] = Dataset.from_dict({"text": docs, "bucket": [name] * len(docs)})
        else:
            print(f"NOTE: empty holdout: {p}")
    else:
        print(f"NOTE: missing holdout (skip): {p}")

print("Holdout buckets:", {k: len(v) for k, v in holdouts.items()})
if "puritan" not in holdouts and not ALLOW_SPURGEON_ONLY:
    print("WARNING: puritan holdout empty — domain eval will be weak until data v2 is complete.")

## 5. Save to disk

In [ ]:
import shutil

if os.path.exists(OUT_TRAIN):
    shutil.rmtree(OUT_TRAIN)
split.save_to_disk(OUT_TRAIN)
print("Saved", OUT_TRAIN)

os.makedirs(OUT_HOLDOUTS, exist_ok=True)
for name, ds in holdouts.items():
    out = os.path.join(OUT_HOLDOUTS, name)
    if os.path.exists(out):
        shutil.rmtree(out)
    ds.save_to_disk(out)
    print("Saved", out)

print("\nDone. Version /kaggle/working as Kaggle dataset: theology-cpt-dataset")
print("Mount that dataset into B_training_sota.ipynb")